# Experiment Runner: QRC-ESN vs. Classical ESN

### **Notebook Objective**

This notebook serves as the main script to run a series of experiments comparing two reservoir computing models:
1.  **Quantum Reservoir Computer (QRC-ESN)**
2.  **Classical Echo State Network (ESN)**

### **Process**
- Defines the experiment configuration (data profiles, hyperparameter grids, and constants).
- Iterates through each defined data profile.
- For each profile, it performs a grid search over the hyperparameter space for both models, using parallel processing (`joblib`) for efficiency.
- It saves all collated results into a single `.csv` file for later analysis in a separate notebook.

In [1]:
# === IMPORTS AND SETUP ===
import sys
import os

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import our custom modules from the 'src' directory
from src.data_generation import mackey_glass, generate_arma_data, generate_narma_data
from src.experiment import run_qrc_experiment_with_cv, run_classical_experiment_with_cv

import itertools
import pandas as pd
from joblib import Parallel, delayed
from tqdm.notebook import tqdm


# Jupyter magic command for automatic reloading of external modules
%load_ext autoreload
%autoreload 2

print("Libraries and modules loaded successfully.")

Libraries and modules loaded successfully.


In [2]:
# === EXPERIMENT CONFIGURATION ===

# General constants
SEED = 2025
TRAIN_FRACTION = 0.8  # CV pool fraction (remaining 20% is held-out test)
N_SPLITS = 5          # sliding-window CV folds
RESULTS_FILENAME = '../data/results_comparative.csv' # Results will be saved in the data folder

# Data profiles for analysis
data_profiles = [
    {'name': 'Mackey_Glass_(tau=17)', 'generator': mackey_glass, 'params': {'tau': 17}},
    {'name': 'Mackey_Glass_(tau=30)', 'generator': mackey_glass, 'params': {'tau': 30}},
    {'name': 'Mackey_Glass_(tau=100)', 'generator': mackey_glass, 'params': {'tau': 100}},
    {'name': 'ARMA_1_2_stochastic', 'generator': generate_arma_data, 'params': {}},
    {'name': 'NARMA10_Chaotic', 'generator': generate_narma_data, 'params': {'order': 10}},
    {'name': 'NARMA5_Chaotic', 'generator': generate_narma_data, 'params': {'order': 5}}
]

# --- Hyperparameter grid for the QRC-ESN model (slim) ---
# n_layers fixed at 2: empirically optimal in 5/6 profiles in the previous run
# and theoretically motivated as the minimal depth introducing non-trivial
# entanglement in the CNOT chain without circuit-scrambling artifacts.
# leakage_rate=1.0 added as an ablation: classical state = current input window,
# isolating the contribution of pure sliding-window memory (no leaky integration).
param_grid_qrc = {
    'leakage_rate': [0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
    'lambda_reg': [1e-8],
    'window_size': [1, 2, 4, 6, 8, 10],
    'n_layers': [2],
    'lag': [0]
}

# --- Hyperparameter grid for the Classical ESN model (slim) ---
# Structural parameters fixed at literature defaults (Jaeger-style ESN baseline):
#   reservoir_size = 100   — middle of the previous grid, standard go-to value
#   spectral_radius = 0.9  — guarantees Echo State Property, most common default
#   sparsity = 0.1         — code's convention (= fraction zeroed); 90% dense reservoir
# leakage_rate is searched with 1.0 included as ablation (analogous to QRC).
# window_size is now searched on the SAME grid as QRC for fair comparability —
# previously hardcoded at 10, which biased the comparison toward QRC at small windows.
param_grid_classical = {
    'reservoir_size': [100],
    'spectral_radius': [0.9],
    'sparsity': [0.1],
    'leakage_rate': [0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
    'lambda_reg': [1e-8],
    'window_size': [1, 2, 4, 6, 8, 10]
}

print(f"Configuration ready. Results will be saved to: {RESULTS_FILENAME}")
print(f"QRC grid size per profile:        {6*1*6*1*1} combinations")
print(f"Classical grid size per profile:  {1*1*1*6*1*6} combinations")

Configuration ready. Results will be saved to: ../data/results_comparative.csv
QRC grid size per profile:        36 combinations
Classical grid size per profile:  36 combinations


In [3]:
# === MAIN EXECUTION LOOP ===
# For every hyperparameter combination we run K-fold sliding-window CV (val MSE)
# AND a Variant-A test evaluation (one re-training on the full CV pool per sub-seed).
# This populates median_test_mse for every row so the CV-vs-Test scatter and PCA plots
# can use all combinations, not just winners.

all_results = []

for profile in data_profiles:
    print(f"{'='*20}STARTING PROFILE: {profile['name']}{'='*20}")
    time_series = profile['generator'](**profile['params'])

    # --- QRC Grid Search ---
    param_combinations_qrc = list(itertools.product(*param_grid_qrc.values()))
    qrc_results = Parallel(n_jobs=-1)(
        delayed(run_qrc_experiment_with_cv)(params, profile, time_series, TRAIN_FRACTION, N_SPLITS, SEED)
        for params in tqdm(param_combinations_qrc, desc=f"QRC Grid Search ({profile['name']})")
    )
    all_results.extend(filter(None, qrc_results))

    # --- Classical ESN Grid Search ---
    param_combinations_classical = list(itertools.product(*param_grid_classical.values()))
    classical_results = Parallel(n_jobs=-1)(
        delayed(run_classical_experiment_with_cv)(params, profile, time_series, TRAIN_FRACTION, N_SPLITS, SEED)
        for params in tqdm(param_combinations_classical, desc=f"Classical ESN Search ({profile['name']})")
    )
    all_results.extend(filter(None, classical_results))

print("--- ALL EXPERIMENTS COMPLETED ---")

====================STARTING PROFILE: Mackey_Glass_(tau=17)====================


QRC Grid Search (Mackey_Glass_(tau=17)):   0%|          | 0/36 [00:00<?, ?it/s]

Classical ESN Search (Mackey_Glass_(tau=17)):   0%|          | 0/36 [00:00<?, ?it/s]

====================STARTING PROFILE: Mackey_Glass_(tau=30)====================


QRC Grid Search (Mackey_Glass_(tau=30)):   0%|          | 0/36 [00:00<?, ?it/s]

...

Classical ESN Search (NARMA10_Chaotic):   0%|          | 0/36 [00:00<?, ?it/s]

====================STARTING PROFILE: NARMA5_Chaotic====================


QRC Grid Search (NARMA5_Chaotic):   0%|          | 0/36 [00:00<?, ?it/s]

Classical ESN Search (NARMA5_Chaotic):   0%|          | 0/36 [00:00<?, ?it/s]

--- ALL EXPERIMENTS COMPLETED ---


In [4]:
# === SAVE RESULTS ===

# Convert the list of dictionaries to a pandas DataFrame
results_df = pd.DataFrame(all_results)

# Save the DataFrame to a CSV file
results_df.to_csv(RESULTS_FILENAME, index=False)

print(f"Successfully saved {len(results_df)} results to {RESULTS_FILENAME}")
results_df.head()

Successfully saved 432 results to ../data/results_comparative.csv


,model_type,data_profile,median_cv_mse,std_cv_mse,cv_cv_mse,median_test_mse,std_test_mse,cv_test_mse,leakage_rate,lambda_reg,window_size,n_layers,lag,base_seed,reservoir_size,spectral_radius,sparsity
0,QRC,Mackey_Glass_(tau=17),0.038670,0.001679,0.043429,0.037599,1.355872e-17,3.606121e-16,0.1,1.000000e-08,1,2.0,0.0,2025,NaN,NaN,NaN
1,QRC,Mackey_Glass_(tau=17),0.000304,0.000077,0.251798,0.000291,5.560412e-05,1.913289e-01,0.1,1.000000e-08,2,2.0,0.0,2025,NaN,NaN,NaN
2,QRC,Mackey_Glass_(tau=17),0.000042,0.000045,1.072764,0.000041,3.653136e-05,8.875206e-01,0.1,1.000000e-08,4,2.0,0.0,2025,NaN,NaN,NaN
3,QRC,Mackey_Glass_(tau=17),0.000008,0.000008,0.933158,0.000007,5.964469e-06,8.732474e-01,0.1,1.000000e-08,6,2.0,0.0,2025,NaN,NaN,NaN
4,QRC,Mackey_Glass_(tau=17),0.000008,0.000009,1.151143,0.000006,7.189453e-06,1.267298e+00,0.1,1.000000e-08,8,2.0,0.0,2025,NaN,NaN,NaN
